# 02 — FOX News Article Data Cleaning & EDA

This notebook will load the scraped FOX news articles, clean up rows if necessary, and explore features of the articles before modeling.

**Note**

After examining the original outputted FOX news articles dataset, I found several bugs that were causing a significant number of NA values. While I did use this notebook to explore them, I ultimately decided to go back to the scraper and make fixes there, so the process of diagnosing and fixing the bugs will not be shown in this notebook. Instead, I'll document my findings here for parity:

1. There were 443 urls that ended with `.print` that were returning invalid content with run with `requests.get` --> I changed the load_links() function to automatically strip `.print` off of the end of the urls before feeding them into the `main()` function.
2. There were 10 urls that came from `foxnews.com/live-news`. The webpages for `/live-news` differ significantly from the other webpages and broke the scraper's logic --> I created a dedicated function for scraping those links and built in an if statement to route those links to the new scraping function
3. There were 11 urls that simply did not have a subtitle. These rows were left as-is since they were indicative of the scraper functioning as intended. 

## Load data

In this section I'll do some basic NA checks and make sure that any remaining NA values are legitimate. 

In [98]:
import pandas as pd

In [99]:
headlines = pd.read_csv("../data/raw/fox_scraped_all.csv")

In [100]:
headlines.isna().sum()

url                 0
topic               0
title               0
subtitle           13
author              0
datetime_posted     0
label               0
dtype: int64

In [101]:
headlines[headlines["subtitle"].isna()]["url"].values

<StringArray>
[                                                                'https://www.foxnews.com/science/discovery-civil-war-map-antietam',
 'https://www.foxnews.com/entertainment/lori-loughlin-mossimo-giannulli-officially-plead-guilty-in-college-admissions-scandal-case',
                  'https://www.foxnews.com/politics/obama-stumping-harris-key-battleground-charges-trump-will-makes-problems-worse',
                                                   'https://www.foxnews.com/opinion/massive-tax-increases-coming-second-biden-term',
                                         'https://www.foxnews.com/entertainment/arnold-schwarzenegger-gruesome-injury-total-recall',
                                                               'https://www.foxnews.com/media/missing-korean-war-airman-fox-nation',
                                          'https://www.foxnews.com/entertainment/olivia-jade-influencer-brand-dumpster-fire-expert',
                                                      '

After going through all 13 links, each of the 13 NA values in `subtitle` were caused by articles without a subtitle.

### Adding a better topic column

In [102]:
headlines["topic"].unique()

<StringArray>
[        'Secretary of State',                    'Divorce',
                      'Media',             'Celebrity News',
                  'LIFESTYLE',                   'POLITICS',
                'White House',                 'Media Buzz',
                  'Exclusive',                     'Israel',
 ...
                   'Homicide',          'National Security',
             'British Royals',                       'Iowa',
                      'Style',                     'Nascar',
                     'Oscars',                   'Odd News',
 'College Admissions Scandal',              'Lent and Life']
Length: 314, dtype: str

In [104]:
headlines["topic"].value_counts()[1:10]

topic
POLITICS         196
LIFESTYLE        148
Donald Trump      76
Israel            58
HEALTH            55
ENTERTAINMENT     53
Kamala Harris     53
Joe Biden         38
OPINION           38
Name: count, dtype: int64

It looks like the "topic" pills at the top of the fox articles were not consistent topic classifiers but rather just "related descriptors". 314 unique "topics" doesn't seem directly useful so I'm going to create a secondary topic column that extracts the article topic from its url. One interesting finding from the original topic column is that "Donald Trump" and "Israel" are the fourth and fifth most fequently listed "topic" in the dataset, above "HEALTH", "ENTERTAINMENT", and "OPINION". Since we don't know how these article links are acquired we can't really speak to whether this is reflective of Fox News' overall coverage, though.

In [105]:
headlines["category"] = headlines["url"].str.extract(r"foxnews\.com/([^/]+)")
headlines["category"].value_counts()

category
politics          615
media             385
lifestyle         203
us                201
entertainment     166
world             110
sports             90
health             73
travel             41
opinion            39
food-drink         35
official-polls     14
tech               12
live-news          10
faith-values        3
science             1
auto                1
great-outdoors      1
Name: count, dtype: int64

## Basic EDA

In [118]:
print("BASIC DF INFO")
print("----------------")
print(f"Df columns: [{", ".join(headlines.columns)}]")
print(f"\nDf shape: {headlines.shape[0]} rows, {headlines.shape[1]} columns")
print(f"\nDf dtypes: \n{headlines.dtypes}")
print(f"\nDf null values:\n{headlines.isna().sum()}")

BASIC DF INFO
----------------
Df columns: [url, topic, title, subtitle, author, datetime_posted, label, category]

Df shape: 2000 rows, 8 columns

Df dtypes: 
url                str
topic              str
title              str
subtitle           str
author             str
datetime_posted    str
label              str
category           str
dtype: object

Df null values:
url                 0
topic               0
title               0
subtitle           13
author              0
datetime_posted     0
label               0
category            0
dtype: int64


In [122]:
headlines["datetime_posted"]

0       2023-10-13T14:06:08-04:00
1       2024-10-18T15:56:05-04:00
2       2024-08-19T21:00:35-04:00
3       2023-06-09T13:55:28-04:00
4       2024-06-06T04:30:24-04:00
                  ...            
1995    2024-09-18T18:00:01-04:00
1996    2024-09-29T04:00:27-04:00
1997    2024-10-16T15:12:35-04:00
1998    2020-09-21T20:39:36-04:00
1999    2024-09-08T04:00:18-04:00
Name: datetime_posted, Length: 2000, dtype: str

In [123]:
headlines["author"].value_counts()

author
Joseph A. Wulfsohn                                                                                                                               69
Paul Steinhauser                                                                                                                                 66
Kerry J. Byrne                                                                                                                                   63
Melissa Rudy                                                                                                                                     51
Julia Johnson                                                                                                                                    47
                                                                                                                                                 ..
Doreen Denny                                                                                             